In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# 1. Load Data
try:
    df = pd.read_csv('zomato.csv', encoding='latin-1')
except FileNotFoundError:
    raise Exception("zomato.csv not found. Download from Kaggle and place in directory.")

# 2. Atomic Cleaning
initial_rows = len(df)
drop_cols = [c for c in ['url', 'address', 'phone', 'menu_item', 'dish_liked', 'reviews_list'] if c in df.columns]
df.drop(columns=drop_cols, inplace=True, errors='ignore')

if 'rate' in df.columns:
    df['rate'] = df['rate'].astype(str).apply(lambda x: x.split('/')[0] if '/' in str(x) else x)
    df['rate'] = pd.to_numeric(df['rate'], errors='coerce')

if 'cost_for_two' in df.columns:
    df['cost_for_two'] = pd.to_numeric(df['cost_for_two'], errors='coerce')

critical = [c for c in ['rate', 'cost_for_two', 'cuisines'] if c in df.columns]
df.dropna(subset=critical, inplace=True)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

# 3. Feature Engineering
if 'location' in df.columns:
    df['Location'] = df['location'].str.strip()
elif 'locality' in df.columns:
    df['Location'] = df['locality'].str.strip()
else:
    df['Location'] = 'Unknown'

if 'cuisines' in df.columns:
    df['Primary_Cuisine'] = df['cuisines'].apply(lambda x: str(x).split(',')[0].strip())

# 4. Analysis Metrics
loc_counts = df['Location'].value_counts().head(10)
avg_rating = df['rate'].mean() if 'rate' in df.columns else 0
top_cuisine = df['Primary_Cuisine'].value_counts().index[0] if 'Primary_Cuisine' in df.columns else 'N/A'
best_loc = df.groupby('Location')['rate'].mean().sort_values(ascending=False).head(1)
corr = df['cost_for_two'].corr(df['rate']) if 'cost_for_two' in df.columns and 'rate' in df.columns else 0

# 5. Visualization Generation
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Chart 1: Top Locations
axes[0, 0].barh(loc_counts.index, loc_counts.values, color='#2ecc71')
axes[0, 0].set_title('Top 10 Locations by Outlet Count')
axes[0, 0].invert_yaxis()

# Chart 2: Cuisine Distribution
if 'Primary_Cuisine' in df.columns:
    top_cuis = df['Primary_Cuisine'].value_counts().head(8)
    colors = plt.cm.Set2(np.linspace(0, 1, len(top_cuis)))
    axes[0, 1].pie(top_cuis.values, labels=top_cuis.index, autopct='%1.1f%%', colors=colors)
    axes[0, 1].set_title('Top 8 Cuisine Types')

# Chart 3: Rating Box Plot
if 'rate' in df.columns:
    top_loc_names = df['Location'].value_counts().head(8).index
    plot_data = df[df['Location'].isin(top_loc_names)]
    bp = axes[1, 0].boxplot([plot_data[plot_data['Location']==l]['rate'].dropna() for l in top_loc_names],
                            labels=top_loc_names, patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.viridis(np.linspace(0, 1, len(top_loc_names)))):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[1, 0].tick_params(axis='x', rotation=45)
    axes[1, 0].set_title('Rating Distribution by Top Locations')

# Chart 4: Cost vs Rating
if 'cost_for_two' in df.columns and 'rate' in df.columns:
    valid = df[['cost_for_two', 'rate']].dropna()
    axes[1, 1].scatter(valid['cost_for_two'], valid['rate'], alpha=0.3, c=valid['rate'], cmap='YlOrRd', s=50)
    z = np.polyfit(valid['cost_for_two'], valid['rate'], 1)
    p = np.poly1d(z)
    axes[1, 1].plot(valid['cost_for_two'], p(valid['cost_for_two']), "r--")
    axes[1, 1].set_title('Cost vs Rating Correlation')

plt.tight_layout()
plt.savefig('restaurant_analysis.png', dpi=300, bbox_inches='tight')
plt.close()

# 6. Final Report Generation
report = f"""ANALYSIS REPORT
----------------
Dataset Size: {len(df)} rows
Top Location: {loc_counts.index[0]} ({loc_counts.iloc[0]} outlets)
Average Rating: {avg_rating:.2f}
Top Cuisine: {top_cuisine}
Best Rated Area: {best_loc.index[0]} ({best_loc.values[0]:.2f})
Cost-Rating Correlation: {corr:.3f}

CONCLUSION:
Market saturation is highest in {loc_counts.index[0]}. 
Ratings cluster around {avg_rating:.2f}, indicating limited differentiation.
Correlation between price and quality is {'negligible' if abs(corr) < 0.1 else 'present'}.
"""

with open('analysis_conclusion.txt', 'w') as f:
    f.write(report)